In [1]:
import os

# Clear Kaggle TPU environment variables before importing torch_xla
os.environ.pop('TPU_PROCESS_ADDRESSES', None)
os.environ.pop('CLOUD_TPU_TASK_ID', None)

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import timm

SCALES_TO_RUN = ["10%", "25%", "50%", "100%"]
EPOCHS        = 200     # Maximum cap; early stopping prevents overfitting on small scales
PATIENCE      = 20      # Number of epochs to wait for validation accuracy improvement
LR            = 5e-4
WEIGHT_DECAY  = 0.05

TEMPERATURE   = 4.0
W_TASK        = 0.6     # Weight for hard cross-entropy task loss
W_DISTILL     = 0.4     # Weight for soft KL divergence distillation loss

PROJ_DIM_IN   = 192
PROJ_DIM_OUT  = 2048

In [2]:
DEVICE_TYPE = "TPU"   # "GPU" | "TPU"

class DeviceManager:
    """Unified stub so the training loop code handles both GPU and TPU seamlessly."""
    def __init__(self):
        self.type = DEVICE_TYPE
        if self.type == "TPU":
            import torch_xla
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr
            import torch_xla.distributed.parallel_loader as pl
            self.xm = xm
            self.xr = xr
            self.pl = pl
            self.device = torch_xla.device()
            self.world_size = xr.world_size()
            self.rank = xr.global_ordinal()
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.world_size = 1
            self.rank = 0

    def step(self, opt):
        if self.type == "TPU":
            self.xm.optimizer_step(opt)
            self.xm.mark_step()
        else:
            opt.step()

    def wrap_loader(self, loader):
        if self.type == "TPU":
            return self.pl.MpDeviceLoader(loader, self.device)
        return loader

    def is_master(self):
        return self.rank == 0

    def master_print(self, *args, **kwargs):
        if self.is_master():
            print(*args, **kwargs)

In [3]:
TRAIN_DIR    = "/kaggle/input/datasets/melikechan/cifar100/cifar100/train"
TEST_DIR     = "/kaggle/input/datasets/melikechan/cifar100/cifar100/test"
TEACHER_CKPT = "/kaggle/input/datasets/totallyapoorv/resnet-teachermodel-50/teacher_resnet50.pth"

WORK_DIR = "./outputs"
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",     exist_ok=True)

In [4]:
SEED       = 67
BATCH_SIZE = 64
scale_map  = {"10%": 0.10, "25%": 0.25, "50%": 0.50, "100%": 1.0}
key_of     = lambda s: s.replace("%", "pct")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

def build_loaders(scale, ctx):
    tfm_tr = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    tfm_te = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    
    full = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=tfm_tr)
    test = torchvision.datasets.ImageFolder(TEST_DIR,  transform=tfm_te)
    
    idx = list(range(len(full)))
    random.Random(SEED).shuffle(idx)
    sub = torch.utils.data.Subset(full, idx[:int(len(full) * scale_map[scale])])
    
    sampler = torch.utils.data.distributed.DistributedSampler(
        sub, num_replicas=ctx.world_size, rank=ctx.rank, shuffle=True
    ) if ctx.world_size > 1 else None

    tr = torch.utils.data.DataLoader(
        sub, BATCH_SIZE, shuffle=(sampler is None), 
        sampler=sampler, num_workers=2, pin_memory=True
    )
    te = torch.utils.data.DataLoader(test, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, te, len(sub)

In [5]:
def evaluate(model, loader, ctx):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in ctx.wrap_loader(loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            out = model(x)
            if isinstance(out, tuple): 
                out = (out[0] + out[1]) / 2
            elif hasattr(out, 'cls_logits') and hasattr(out, 'dist_logits'):
                out = (out.cls_logits + out.dist_logits) / 2
            correct += out.argmax(1).eq(y).sum().item()
            total   += y.size(0)
    
    if ctx.type == "TPU":
        correct = ctx.xm.mesh_reduce("test_correct", correct, sum)
        total = ctx.xm.mesh_reduce("test_total", total, sum)
        
    return 100. * correct / total

def save_ckpt(path, epoch, model, proj, opt, best_acc, history, extras=None):
    d = dict(epoch=epoch, model=model.state_dict(), opt=opt.state_dict(),
             best_acc=best_acc, history=history)
    if proj is not None: d["proj"] = proj.state_dict()
    if extras:           d.update(extras)
    torch.save(d, path)

def load_ckpt(path, model, proj, opt, ctx):
    # Forced 'cpu' mapping prevents UntypedStorage xla:0 tagging crashes when parsing resume files
    d  = torch.load(path, map_location='cpu')
    model.load_state_dict(d["model"])
    if proj is not None and "proj" in d: proj.load_state_dict(d["proj"])
    opt.load_state_dict(d["opt"])
    return d["epoch"]+1, d["best_acc"], d.get("history", [])

In [6]:
def build_teacher(ctx):
    t = torchvision.models.resnet50(weights=None)
    t.fc = nn.Linear(t.fc.in_features, 100)
    # Changed map_location from ctx.device to 'cpu' to unpack the storage layout properly
    t.load_state_dict(torch.load(TEACHER_CKPT, map_location='cpu'))
    t = t.to(ctx.device).eval()
    for p in t.parameters(): 
        p.requires_grad_(False)
    return t

def build_student(ctx):
    m = timm.create_model("deit_tiny_distilled_patch16_224", 
                          pretrained=False, num_classes=100)
    m.set_distilled_training(True)
    m = m.to(ctx.device)
    if ctx.is_master():
        total_params = sum(p.numel() for p in m.parameters())
        print(f"  Student params: {total_params/1e6:.2f}M")
    return m

def log_teacher_accuracy(teacher, te_loader, ctx):
    rp = f"{WORK_DIR}/results/teacher_accuracy.json"
    if os.path.exists(rp):
        if ctx.is_master():
            r = json.load(open(rp))
            ctx.master_print(f"Teacher accuracy already computed: {r['acc']:.2f}%")
        return

    teacher.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in ctx.wrap_loader(te_loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            correct += teacher(x).argmax(1).eq(y).sum().item()
            total += y.size(0)
            
    if ctx.type == "TPU":
        correct = ctx.xm.mesh_reduce("teacher_correct", correct, sum)
        total = ctx.xm.mesh_reduce("teacher_total", total, sum)

    acc = 100. * correct / total
    if ctx.is_master():
        json.dump({"acc": acc}, open(rp, "w"))
        ctx.master_print(f"Teacher (ResNet-50) accuracy: {acc:.2f}%")

In [7]:
def kd_loss(cls_logits, dist_logits, teacher_logits, targets, T=TEMPERATURE):
    task    = F.cross_entropy(cls_logits, targets)
    soft_t  = F.softmax(teacher_logits / T, dim=1)
    log_s   = F.log_softmax(dist_logits  / T, dim=1)
    distill = F.kl_div(log_s, soft_t, reduction="batchmean") * (T ** 2)
    kl_mag  = distill.detach().item()
    
    combined_loss = (W_TASK * task) + (W_DISTILL * distill)
    return combined_loss, task.item(), distill.item(), kl_mag

In [8]:
def train_one_scale(index, scale):
    ctx = DeviceManager()
    key  = key_of(scale)
    ckpt = f"{WORK_DIR}/checkpoints/nb02_{key}.pth"
    rp   = f"{WORK_DIR}/results/nb02_{key}.json"

    if os.path.exists(rp) and json.load(open(rp)).get("completed"):
        r = json.load(open(rp))
        ctx.master_print(f"[{scale}] already done — best {r['best_acc']:.2f}%")
        return r

    tr_loader, te_loader, n_train = build_loaders(scale, ctx)
    
    teacher = build_teacher(ctx)
    log_teacher_accuracy(teacher, te_loader, ctx)
    
    student = build_student(ctx)
    opt     = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    start_epoch, best_acc, history = 0, 0.0, []
    epochs_no_improve = 0
    
    if os.path.exists(ckpt):
        start_epoch, best_acc, history = load_ckpt(ckpt, student, None, opt, ctx)
        ctx.master_print(f"[{scale}] resumed at epoch {start_epoch+1}")

    ctx.master_print(f"[{scale}] training on {n_train} images")
    for epoch in range(start_epoch, EPOCHS):
        if hasattr(tr_loader, 'sampler') and hasattr(tr_loader.sampler, 'set_epoch'):
            tr_loader.sampler.set_epoch(epoch)
            
        student.train()
        t0 = time.time()
        tot_loss = task_sum = distill_sum = 0.0

        for x, y in ctx.wrap_loader(tr_loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            opt.zero_grad()
            
            with torch.no_grad():
                t_logits = teacher(x)
                
            out = student(x)
            
            if isinstance(out, tuple):
                cls_l, dist_l = out
            elif hasattr(out, 'cls_logits') and hasattr(out, 'dist_logits'):
                cls_l, dist_l = out.cls_logits, out.dist_logits
            else:
                cls_l, dist_l = out, out
                
            loss, tl, dl, _ = kd_loss(cls_l, dist_l, t_logits, y)
            loss.backward()
            ctx.step(opt)
            
            tot_loss    += loss.item()
            task_sum    += tl
            distill_sum += dl  # Fixed variable name mismatch here

        epoch_time = time.time() - t0
        scheduler.step()
        
        nb_ = len(tr_loader)
        if ctx.type == "TPU":
            avg_loss = ctx.xm.mesh_reduce("loss_reduce", tot_loss, sum) / (nb_ * ctx.world_size)
            avg_task = ctx.xm.mesh_reduce("task_reduce", task_sum, sum) / (nb_ * ctx.world_size)
            avg_dist = ctx.xm.mesh_reduce("dist_reduce", distill_sum, sum) / (nb_ * ctx.world_size)
        else:
            avg_loss = tot_loss / nb_
            avg_task = task_sum / nb_
            avg_dist = distill_sum / nb_
            
        val_acc = evaluate(student, te_loader, ctx)
        
        if val_acc > best_acc:
            best_acc = val_acc
            epochs_no_improve = 0
            is_best = True
        else:
            epochs_no_improve += 1
            is_best = False

        if ctx.is_master():
            rec = {"epoch": epoch, "total_loss": avg_loss, "task_loss": avg_task,
                   "distill_loss": avg_dist, "val_acc": val_acc, "epoch_time": epoch_time}
            history.append(rec)
            ctx.master_print(f"[{scale}] E{epoch+1}/{EPOCHS} | total={avg_loss:.4f} "
                             f"task={avg_task:.4f} kl={avg_dist:.4f} | val={val_acc:.2f}% | "
                             f"{epoch_time:.0f}s | wait={epochs_no_improve}/{PATIENCE}")
            if is_best:
                save_ckpt(ckpt, epoch, student, None, opt, best_acc, history)

        if epochs_no_improve >= PATIENCE:
            ctx.master_print(f"[{scale}] Early stopping triggered at epoch {epoch+1}!")
            break

    if ctx.is_master():
        result = {"scale": scale, "n_train": n_train, "final_acc": history[-1]["val_acc"] if history else best_acc,
                  "best_acc": best_acc, "history": history, "completed": True}
        json.dump(result, open(rp, "w"), indent=2)
        ctx.master_print(f"[{scale}] DONE — final {result['final_acc']:.2f}% | best {best_acc:.2f}%")
        return result
    return None

In [9]:
def _mp_fn(index, scales):
    for scale in scales:
        train_one_scale(index, scale)

if __name__ == "__main__":
    if DEVICE_TYPE == "TPU":
        import torch_xla.distributed.xla_multiprocessing as xmp
        xmp.spawn(_mp_fn, args=(SCALES_TO_RUN,), start_method='fork')
    else:
        _mp_fn(0, SCALES_TO_RUN)
    print("All scales complete.")

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

[100%] E114/200 | total=0.2489 task=0.0021 kl=0.6191 | val=49.85% | 26s | wait=19/20

[100%] E115/200 | total=0.2460 task=0.0021 kl=0.6120 | val=49.98% | 26s | wait=20/20

[100%] Early stopping triggered at epoch 115!

[100%] DONE — final 49.98% | best 50.10%

All scales complete.


In [10]:
print(f"\n{'Scale':<8} {'Final Acc':>10} {'Best Acc':>10} {'Train (s)':>12}")
for scale in SCALES_TO_RUN:
    rp = f"{WORK_DIR}/results/nb02_{key_of(scale)}.json"
    if os.path.exists(rp):
        r = json.load(open(rp))
        wall = sum(h.get("epoch_time", 0) for h in r["history"])
        print(f"{scale:<8} {r['final_acc']:>9.2f}% {r['best_acc']:>9.2f}% {wall:>11.0f}s")
    else:
        print(f"{scale:<8} not yet complete")


Scale     Final Acc   Best Acc    Train (s)
10%          17.79%     18.02%         358s
25%          27.80%     28.36%         465s
50%          39.11%     39.34%        1542s
100%         49.98%     50.10%        3037s
